# Telco Churn Scoring — Deep Dive Tecnico
## Clasificacion, calibracion, lift/gains y priorizacion comercial unificada

**Autor:** Juan Prada | **Dataset:** [Telco Customer Churn — Kaggle](https://www.kaggle.com/blastchar/telco-customer-churn) · 7,043 clientes · 21 variables

---

## Que cubre este notebook

1. Setup e imports
2. Carga y validacion (desbalanceo 26.5%)
3. EDA
4. Feature engineering + split train/val/test
5. Hyperparameter tuning (solo sobre train)
6. Calibracion de probabilidades + metricas (F1, PR-AUC, Brier, ECE)
7. Threshold por matriz de costes + lift/gains
8. Interpretabilidad (feature importance / SHAP)
9. Scoring de churn + Case 2 (potencial) + Case 3 (anomalias)
10. Score comercial unificado
11. Conclusiones


## 1. Setup e imports


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_data, validate_data
from src.features.engineering import preprocess, build_features, split_data
from src.models.train import (
    train_evaluate_with_calibration,
    tune_threshold_cost,
    build_cost_curve,
    build_churn_scoring,
)
from src.models.tuning import tune_all_models
from src.models.lift import run_lift_analysis
from src.cases import commercial_potential, anomaly_detection, unified_scoring
from src.visualization.plots import (
    plot_churn_distribution,
    plot_numeric_by_churn,
    plot_categorical_churn_rate,
    plot_roc_curves,
    plot_pr_curves,
    plot_calibration_curve,
    plot_cost_curve,
    plot_feature_importance,
    plot_shap_summary,
    plot_churn_score_distribution,
    plot_lift_gains,
    plot_potential_scoring,
    plot_anomaly_scoring,
    plot_unified_scoring,
)

sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 261
DATA_PATH = Path("../data/telco_churn.csv")
REPORTS = Path("../output/reports")
REPORTS.mkdir(parents=True, exist_ok=True)

print("Entorno configurado")

## 2. Carga y validacion de datos

El dataset esta **desbalanceado de forma moderada**: ~26.5% churn.
Accuracy no sirve (un modelo naive "siempre No Churn" acierta ~73% sin detectar a nadie).
Por eso usamos F1 / PR-AUC / Recall y `class_weight="balanced"`.


In [ ]:
df_raw = load_data(DATA_PATH)
validation_report = validate_data(df_raw)
df_raw.head(3)

In [ ]:
print("Variables numericas:", list(df_raw.select_dtypes("number").columns))
print("\nVariables categoricas:", list(df_raw.select_dtypes("object").columns))
print("\nNota: TotalCharges llega como string. Se convierte en loader.py.")
print("11 clientes con tenure=0 tienen TotalCharges vacio, se imputa a 0.")
print(f"\nChurn rate confirmado: {(df_raw['Churn']=='Yes').mean()*100:.2f}%")

## 3. EDA

### 3.1 Distribucion del target


In [ ]:
plot_churn_distribution(df_raw, save_path=None)

### 3.2 Variables numericas vs Churn


In [ ]:
plot_numeric_by_churn(df_raw, num_cols=["tenure", "MonthlyCharges", "TotalCharges"], save_path=None)

Insights:
- `tenure`: churners con menor antiguedad → clientes nuevos = mayor riesgo
- `MonthlyCharges`: factura alta asociada a mas churn
- `TotalCharges`: correlacionada con tenure (los que se van pronto acumulan menos)


### 3.3 Categoricas de alto impacto


In [ ]:
plot_categorical_churn_rate(
    df_raw,
    cat_cols=["Contract", "InternetService", "PaymentMethod", "TechSupport", "OnlineSecurity"],
    target="Churn",
    save_path=None,
)

Insights:
- `Contract` mes a mes: churn >42%; dos anos: <3% (predictor mas fuerte)
- Fibra optica: mas churn que DSL
- Sin TechSupport / OnlineSecurity: churn mas alto


## 4. Feature engineering + split train/val/test

**Por que train/val/test y no solo train/test?**
- `train`: aprender parametros + tuning de hiperparametros
- `val`: calibrar probabilidades y elegir umbrales (sin tocar test)
- `test`: estimar rendimiento final de forma honesta

Features nuevas relevantes:
- indicadores `has_*` de servicios
- `charge_ratio`, `deviation_from_expected`, `avg_monthly_charge`
- `log_total_charges`, `log_monthly_charges`


In [ ]:
df_processed = preprocess(df_raw)
df_features = build_features(df_processed)

X_train, X_val, X_test, y_train, y_val, y_test, cid_train, cid_val, cid_test = split_data(
    df_features, val_size=0.2, test_size=0.2
)

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight (XGBoost) = {scale_pos:.2f}")
print(f"Features finales ({X_train.shape[1]}):")
print(list(X_train.columns))

## 5. Hyperparameter tuning (solo sobre train)

`RandomizedSearchCV` con 5-fold estratificado **exclusivamente en train**.
Val y test quedan intocados para evitar data leakage de seleccion de modelo.


In [ ]:
tuned_models, tuning_summary = tune_all_models(
    X_train, y_train, scale_pos_weight=scale_pos, n_iter=20, cv=5
)
tuning_summary

## 6. Calibracion + evaluacion

Se calibra con Val (sigmoid / Platt scaling) y se reporta en Test:
- **F1** (metrica principal con desbalanceo)
- **ROC-AUC** y **PR-AUC**
- **Brier** y **ECE** (calidad de la probabilidad)


In [ ]:
results, trained_detail = train_evaluate_with_calibration(
    tuned_models, X_train, y_train, X_val, y_val, X_test, y_test
)
results.to_csv(REPORTS / "model_comparison.csv", index=False)
results

In [ ]:
calibrated_models = {name: d["calibrated"] for name, d in trained_detail.items()}
plot_roc_curves(calibrated_models, X_test, y_test, save_path=None)
plot_pr_curves(calibrated_models, X_test, y_test, save_path=None)

best_name = results.iloc[0]["Model"]
best_calibrated = trained_detail[best_name]["calibrated"]
best_base = trained_detail[best_name]["base"]
print(f"Mejor modelo: {best_name}")
print(results.iloc[0][["F1-Score", "AUC-ROC", "PR-AUC", "Precision", "Recall", "BrierScore", "ECE"]].to_string())

In [ ]:
plot_calibration_curve(best_calibrated, X_test, y_test, model_name=best_name, save_path=None)

## 7. Threshold por costes + Lift / Gains

Con `cost(FN)=5` y `cost(FP)=1` (perder un cliente cuesta 5x mas que una visita fallida),
el umbral optimo cae por debajo de 0.5. Eso es consecuencia directa del desbalanceo + asimetría de costes.

Lift responde a negocio: *"si contacto el top 10%/20% del score, cuantos churners capturo vs aleatorio?"*


In [ ]:
proba_val = best_calibrated.predict_proba(X_val)[:, 1]
cost_result = tune_threshold_cost(y_val, proba_val, cost_fn_fp=1.0, cost_fn_false_neg=5.0)
cost_df = build_cost_curve(y_val, proba_val, cost_fn_fp=1.0, cost_fn_false_neg=5.0)
print(cost_result)
plot_cost_curve(cost_df, optimal_threshold=cost_result["threshold"], save_path=None)

In [ ]:
proba_test = best_calibrated.predict_proba(X_test)[:, 1]
lift_result = run_lift_analysis(y_test, proba_test, model_name=best_name, n_bins=10, output_dir=REPORTS)
display(lift_result["lift_table"])
plot_lift_gains(lift_result["lift_table"], model_name=best_name, save_path=None)

## 8. Interpretabilidad


In [ ]:
if hasattr(best_base, "feature_importances_"):
    plot_feature_importance(best_base, X_train.columns.tolist(), top_n=15, model_name=best_name, save_path=None)
    plot_shap_summary(best_base, X_test, model_name=best_name, save_path=None)
else:
    print("El mejor modelo no expone feature_importances_ directamente (p.ej. LogisticRegression en Pipeline).")

## 9. Scoring de churn + Case 2 + Case 3

### Case 1 — Churn scoring (toda la base)
Tiers dinamicos desde Val (`risk_threshold_medium/high`), no cortes fijos 0.3/0.6.


In [ ]:
X_all = df_features.drop(columns=["Churn", "customerID"])
customer_ids_all = df_features["customerID"]

churn_scoring = build_churn_scoring(
    best_calibrated,
    X_all,
    customer_ids=customer_ids_all,
    threshold_medium=trained_detail[best_name]["risk_threshold_medium"],
    threshold_high=trained_detail[best_name]["risk_threshold_high"],
)
churn_scoring.to_csv(REPORTS / "churn_scoring.csv", index=False)
print(churn_scoring["risk_tier"].value_counts())
plot_churn_score_distribution(
    churn_scoring,
    threshold_medium=trained_detail[best_name]["risk_threshold_medium"],
    threshold_high=trained_detail[best_name]["risk_threshold_high"],
    save_path=None,
)
display(churn_scoring.head(10))

### Case 2 — Potencial comercial (regresion)

Target: `monthly_potential = P75(MonthlyCharges | Contract, InternetService) - MonthlyCharges`.
Ranking de upsell por incremento esperado de facturacion.


In [ ]:
case2 = commercial_potential.run(df_raw, output_dir=REPORTS)
display(case2["results"])
plot_potential_scoring(case2["scoring"], save_path=None)
display(case2["scoring"].head(10))

### Case 3 — Anomalias de facturacion (Isolation Forest)

Unsupervised: no usa Churn como input. Se valida con Precision@K vs base rate
(como chequeo exploratorio, no como objetivo principal).


In [ ]:
case3 = anomaly_detection.run(df_raw, output_dir=REPORTS)
plot_anomaly_scoring(case3["scoring"], save_path=None)
display(case3["scoring"].head(10))

## 10. Score comercial unificado

Combina:
- `churn_score` (peso 0.50)
- `monthly_potential_eur` (peso 0.30)
- `anomaly_score` (peso 0.20)

y asigna un **playbook** (`Retain_HighValue`, `Grow_Upsell`, etc.).


In [ ]:
unified = unified_scoring.run(
    churn_scoring=churn_scoring,
    potential_scoring=case2["scoring"],
    anomaly_scoring=case3["scoring"],
    output_dir=REPORTS,
)
plot_unified_scoring(unified["scoring"], save_path=None)
display(unified["scoring"][[
    "customer_id", "commercial_priority_score", "priority_tier",
    "commercial_segment", "churn_score", "monthly_potential_eur", "anomaly_score",
    "recommended_action",
]].head(15))

## 11. Conclusiones

### Hallazgos tecnicos
- Desbalanceo **26.5%**: accuracy engañosa; F1 + PR-AUC + calibracion son el marco correcto.
- Con `cost(FN)>>cost(FP)` el umbral optimo baja (~0.21 en la ultima ejecucion).
- Lift fuerte: top decile ~**2.8x**; top 20% captura ~**69%** de churners.
- Calibracion (Brier/ECE) habilita interpretar el score como probabilidad de negocio.

### Output operativo
- Case 1: priorizacion de retencion
- Case 2: priorizacion de upsell
- Case 3: alertas de facturacion atipica
- Unificado: una sola cola comercial con playbooks

### Limitaciones
1. Dataset de telecom; revalidar features en otros sectores.
2. Sin dimension temporal (no survival / no validacion temporal).
3. Anomalias detectan rareza de facturacion, no churn directo (Precision@K ~ base rate).
4. Causalidad no garantizada: las acciones deben validarse con A/B.

### Siguiente iteracion sugerida
- Threshold por capacidad real de visitas del equipo
- Optuna / CV anidado mas exhaustivo
- Survival analysis si aparece historico temporal
